# 초파리 뇌 지도 실습 — 구글·자넬리아가 2026년 9월 공개한 MaleCNS
**CONNECT AI LAB · 이론 + 실습 · 준비물: 구글 계정 하나, 비용 0원**

2026년 9월 3일, 자넬리아 연구소·케임브리지 대학·구글 리서치가 수컷 초파리의 **뇌와 척수 전체 배선 지도**를 무료로 공개했습니다. 뉴런 166,700개, 시냅스 연결 약 1억 2,500만 개. 이 노트북에서는 그 지도를 여러분 손으로 직접 열어 봅니다.

| 단계 | 하는 것 | 나오는 것 |
|---|---|---|
| 1 | 뉴런 표를 열어 종류별로 세기 | "다리 운동뉴런이 815개" 같은 문장을 내 손으로 |
| 2 | 뉴런 위치 16만 개를 3D로 그리기 | 논문 그림과 같은 뇌 모양이 내 화면에 |
| 3 | 냄새 뉴런 하나에서 연결을 따라가기 | 코 → 뇌 → 척수 → 다리 근육, 시냅스 몇 단계인지 |
| 4 | 배선을 "물통 모델"로 돌려 보기 | 냄새를 넣으면 어느 뉴런이 발화하는지, 좌우를 바꾸면 어떻게 되는지 |

데이터 출처: https://male-cns.janelia.org/download/ (CC-BY 4.0) · 논문: Berg 외, *Cell* (2026) doi:10.1016/j.cell.2026.08.015

## 0. 준비 — 지도 파일 내려받기
표 세 개를 받습니다. 뉴런 표(13MB)와 신경전달물질 표(42MB)는 금방이고, 연결 표(1.1GB)는 3단계에서 따로 받습니다.
파일 형식은 `feather`입니다. 엑셀보다 훨씬 빨리 열리는 표 형식이고, 코랩에 이미 들어 있는 `pyarrow`가 읽습니다.

In [ ]:
import urllib.request, os
버킷 = "https://storage.googleapis.com/flyem-male-cns/v1.0/connectome-data/flat-connectome/"
파일들 = {
    "뉴런표.feather": "body-annotations-male-cns-v1.0-minconf-0.5.feather",
    "신경전달물질표.feather": "body-neurotransmitters-male-cns-v1.0.feather",
}
for 이름, 원본 in 파일들.items():
    if not os.path.exists(이름):
        urllib.request.urlretrieve(버킷 + 원본, 이름)
    print(이름, round(os.path.getsize(이름)/1e6, 1), "MB")

그림에 한글을 쓰려면 코랩에 한글 글꼴을 한 번 설치합니다(20초). 설치 후 이 셀을 한 번 더 실행하면 한글이 깨지지 않습니다.

In [ ]:
import matplotlib, matplotlib.font_manager as fm, os, subprocess
if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
    for 경로 in ["/usr/share/fonts/truetype/nanum/NanumGothic.ttf"]:
        if os.path.exists(경로): fm.fontManager.addfont(경로)
matplotlib.rc("font", family="NanumGothic" if any("Nanum" in f.name for f in fm.fontManager.ttflist) else "AppleGothic")
matplotlib.rc("axes", unicode_minus=False)
print("그림 글꼴:", matplotlib.rcParams["font.family"])

## 1단계. 뉴런 표 열어보기
표 한 줄이 뉴런 하나입니다(정확히는 전자현미경 사진 속 "덩어리" 하나. 뉴런이 아닌 것도 섞여 있어서 걸러냅니다).
- `superclass`: 큰 분류. 시각엽 내부, 중추뇌 내부, 척수 내부, 하강뉴런(뇌→몸), 운동뉴런…
- `class`: 작은 분류. 후각, 시각, 촉각엽 투사뉴런, 케니언세포, 도파민 뉴런…
- `somaSide`: 세포체가 왼쪽(L)인지 오른쪽(R)인지
- `somaLocation`: 세포체 좌표 (단위: 8나노미터 복셀)

In [ ]:
import pandas as pd
뉴런표 = pd.read_feather("뉴런표.feather")
print("표 크기:", 뉴런표.shape)
뉴런표[["bodyId", "superclass", "class", "type", "somaSide", "somaLocation", "status"]].head(8)

`status`가 `Glia`(신경아교세포, 뉴런 아님)인 줄과 `superclass`가 비어 있는 줄(끊어진 조각)을 빼면 논문의 숫자 166,700이 나옵니다.

In [ ]:
뉴런만 = 뉴런표[(뉴런표["superclass"].notna()) & (뉴런표["status"] != "Glia")].copy()
print("뉴런 개수:", f"{len(뉴런만):,}")

한글이름 = {
    "ol_intrinsic": "시각엽 내부", "cb_intrinsic": "중추뇌 내부", "vnc_intrinsic": "척수 내부",
    "visual_projection": "시각 투사", "vnc_sensory": "척수 감각", "ol_sensory": "눈(광수용)",
    "cb_sensory": "뇌 감각(코·더듬이·입)", "ascending_neuron": "상행(몸→뇌)", "descending_neuron": "하강(뇌→몸)",
    "vnc_motor": "척수 운동뉴런(다리·날개)", "visual_centrifugal": "시각 원심", "sensory_ascending": "감각 상행",
    "cb_motor": "뇌 운동뉴런(주둥이 등)", "endocrine": "내분비",
}
종류별 = 뉴런만["superclass"].map(한글이름).value_counts()
종류별

**여기서 볼 것.** 뉴런의 절반 이상이 시각엽에 있습니다. 초파리는 눈으로 사는 동물입니다. 그리고 이번 지도에만 있는 것, **운동뉴런**. 척수 운동뉴런 708개와 뇌 운동뉴런 107개, 합쳐 815개가 실제로 다리·날개·주둥이 근육을 움직입니다. 2024년 암컷 뇌 지도에는 척수가 없어서 이 뉴런들이 없었습니다.

In [ ]:
print("좌우:", 뉴런만["somaSide"].value_counts(dropna=False).to_dict())
작은분류 = 뉴런만["class"].value_counts()
관심 = {"olfactory": "후각(코)", "visual": "시각(눈)", "mechanosensory": "기계감각(더듬이)", "gustatory": "미각(입)",
        "ALPN": "촉각엽 투사뉴런(냄새 2차 처리)", "Kenyon_Cell": "케니언세포(학습·기억)", "MBON": "버섯체 출력뉴런",
        "DAN": "도파민 뉴런(보상 신호)", "CX": "중심복합체(길찾기)"}
for 영어, 한글 in 관심.items():
    print(f"{한글:<24} {작은분류.get(영어, 0):>6,}개")
운동뉴런수 = (뉴런만["superclass"].isin(["vnc_motor", "cb_motor"])).sum()
print(f"{'운동뉴런(척수+뇌)':<24} {운동뉴런수:>6,}개")

**도파민 뉴런 340개**를 기억해 두세요. 강화학습의 "보상"이 생물학에서 나오는 자리입니다. 이 실습에서는 건드리지 않지만, 다음 편(보상 설계)에서 여기로 돌아옵니다.

## 2단계. 내 손으로 뇌 그리기
세포체 좌표가 있는 뉴런 약 14만 개를 점으로 찍습니다. 좌표 단위가 8나노미터 복셀이라 8을 곱해 나노미터, 1000으로 나눠 마이크로미터로 바꿉니다. 색은 큰 분류별로.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

위치있음 = 뉴런만[뉴런만["somaLocation"].notna()].copy()
좌표 = np.array(위치있음["somaLocation"].tolist(), dtype=float) * 8 / 1000   # µm
색깔표 = {"ol_intrinsic": "#6ba2a1", "cb_intrinsic": "#8eae87", "vnc_intrinsic": "#658376",
        "descending_neuron": "#e9a170", "vnc_motor": "#ffd60a", "cb_sensory": "#d3b965"}
색 = 위치있음["superclass"].map(색깔표).fillna("#9ca3af").tolist()

그림 = plt.figure(figsize=(8, 10), facecolor="#0b0e14")
축 = 그림.add_subplot(111, projection="3d", facecolor="#0b0e14")
축.scatter(좌표[:, 0], 좌표[:, 2], -좌표[:, 1], s=0.2, c=색, alpha=0.6, linewidths=0)
축.set_axis_off(); 축.view_init(elev=8, azim=-90)
축.set_title(f"수컷 초파리 중추신경계 · 세포체 {len(좌표):,}개", color="white")
plt.show()

양쪽에 귀처럼 큰 덩어리가 시각엽(청록), 가운데가 중추뇌(연두), 아래로 길게 내려가는 것이 척수(회녹). 노란 점이 운동뉴런입니다. `azim=-90`을 `0`으로 바꾸면 옆에서 본 모습이 됩니다. 척수가 얼마나 긴지 보세요.

In [ ]:
그림 = plt.figure(figsize=(6, 10), facecolor="#0b0e14")
축 = 그림.add_subplot(111, projection="3d", facecolor="#0b0e14")
축.scatter(좌표[:, 0], 좌표[:, 2], -좌표[:, 1], s=0.2, c=색, alpha=0.6, linewidths=0)
축.set_axis_off(); 축.view_init(elev=8, azim=0)
축.set_title("옆에서 본 모습 — 위가 뇌, 아래가 척수", color="white")
plt.show()

## 3단계. 냄새는 어디로 가나 — 연결 표 따라가기
연결 표는 1.1GB입니다. 1억 5천만 줄이고, 줄 하나가 "뉴런 A → 뉴런 B, 시냅스 n개"입니다. 내려받는 데 1~3분 걸립니다.

In [ ]:
연결원본 = "connectome-weights-male-cns-v1.0-minconf-0.5.feather"
if not os.path.exists("연결표.feather"):
    print("연결 표 내려받는 중 (1.1GB)…")
    urllib.request.urlretrieve(버킷 + 연결원본, "연결표.feather")
print("크기:", round(os.path.getsize("연결표.feather")/1e9, 2), "GB")

1억 5천만 줄을 `pandas`로 열면 메모리가 넘칠 수 있어서, 숫자 배열(`numpy`)로 바로 읽습니다. 그리고 **뉴런끼리의 연결 중 시냅스 10개 이상**만 남깁니다. 약한 연결을 걸러 뼈대만 보는 겁니다(전체 시냅스의 54%가 남습니다).

In [ ]:
import pyarrow.feather as feather
표 = feather.read_table("연결표.feather")
보내는쪽 = 표.column("body_pre").to_numpy().astype(np.int64)
받는쪽   = 표.column("body_post").to_numpy().astype(np.int64)
시냅스수 = 표.column("weight").to_numpy().astype(np.int32)
del 표
print("연결 줄 수:", f"{len(시냅스수):,}")

뉴런번호들 = np.sort(뉴런만["bodyId"].to_numpy().astype(np.int64))
def 뉴런인가(번호배열):
    자리 = np.searchsorted(뉴런번호들, 번호배열)
    자리 = np.minimum(자리, len(뉴런번호들) - 1)
    return 뉴런번호들[자리] == 번호배열

남길것 = 뉴런인가(보내는쪽) & 뉴런인가(받는쪽) & (시냅스수 >= 10)
보내는쪽, 받는쪽, 시냅스수 = 보내는쪽[남길것], 받는쪽[남길것], 시냅스수[남길것]
print("뉴런끼리 · 시냅스 10개 이상 연결:", f"{len(시냅스수):,}")

이제 뉴런 하나를 골라 "누구에게 보내나"를 봅니다. 출발점은 **후각 뉴런**입니다. 후각 뉴런은 세포체가 더듬이(뇌 밖)에 있어서 `somaSide`가 비어 있고, 대신 `rootSide`에 좌우가 있습니다. 유형 이름 `ORN_DP1m`은 발효 냄새(식초·과일)에 반응하는 뉴런입니다. 다른 유형으로 바꿔 보세요.

In [ ]:
정보 = 뉴런만.set_index("bodyId")[["superclass", "class", "type", "rootSide", "somaSide"]]
def 이름표(번호):
    if 번호 not in 정보.index: return "?"
    줄 = 정보.loc[번호]
    return f"{줄['class'] if isinstance(줄['class'], str) else 줄['superclass']}[{줄['type']}]"

후각왼쪽 = 뉴런만[(뉴런만["class"] == "olfactory") & (뉴런만["rootSide"] == "L")]
출발 = int(후각왼쪽[후각왼쪽["type"] == "ORN_DP1m"]["bodyId"].iloc[0])
print("출발 뉴런:", 출발, 이름표(출발))

순서 = np.argsort(보내는쪽, kind="stable")
보내는쪽정렬, 받는쪽정렬, 시냅스정렬 = 보내는쪽[순서], 받는쪽[순서], 시냅스수[순서]
def 연결상대(번호, 몇개=6):
    시작 = np.searchsorted(보내는쪽정렬, 번호); 끝 = np.searchsorted(보내는쪽정렬, 번호, side="right")
    상대 = 받는쪽정렬[시작:끝]; 세기 = 시냅스정렬[시작:끝]
    고른 = np.argsort(-세기)[:몇개]
    return [(int(상대[k]), int(세기[k])) for k in 고른]

print("1단계 연결 상대 (시냅스 많은 순):")
for 번호, 세기 in 연결상대(출발):
    print(f"   시냅스 {세기:>4}개 → {이름표(번호)}")

후각 뉴런은 거의 전부 **촉각엽 투사뉴런(ALPN)**과 촉각엽 국소뉴런(ALLN)으로 보냅니다. 냄새의 첫 정거장입니다. 이제 그 투사뉴런이 어디로 보내는지 한 단계 더 갑니다.

In [ ]:
다음 = [번호 for 번호, _ in 연결상대(출발) if 이름표(번호).startswith("ALPN")][0]
print("2단계:", 이름표(다음), "→")
for 번호, 세기 in 연결상대(다음):
    print(f"   시냅스 {세기:>4}개 → {이름표(번호)}")

이번엔 **최단 경로**를 찾습니다. 후각 뉴런에서 출발해 연결을 따라가며 "뇌에서 몸으로 나가는 하강뉴런"과 "다리 근육을 움직이는 운동뉴런"에 몇 단계 만에 닿는지. 컴퓨터 과학의 너비 우선 탐색(BFS)입니다. 길찾기 앱이 쓰는 것과 같은 방법입니다.

In [ ]:
from collections import defaultdict, deque
이웃 = defaultdict(list)
for a, b in zip(보내는쪽.tolist(), 받는쪽.tolist()):
    이웃[a].append(b)

하강뉴런 = set(뉴런만[뉴런만["superclass"] == "descending_neuron"]["bodyId"].astype(int))
운동뉴런 = set(뉴런만[뉴런만["superclass"].isin(["vnc_motor", "cb_motor"])]["bodyId"].astype(int))

def 최단경로(출발, 목표집합):
    온곳 = {출발: None}; 줄 = deque([출발])
    while 줄:
        지금 = 줄.popleft()
        if 지금 in 목표집합 and 지금 != 출발:
            경로 = []; x = 지금
            while x is not None: 경로.append(x); x = 온곳[x]
            return 경로[::-1]
        for 다음 in 이웃.get(지금, []):
            if 다음 not in 온곳: 온곳[다음] = 지금; 줄.append(다음)

경로1 = 최단경로(출발, 하강뉴런)
경로2 = 최단경로(출발, 운동뉴런)
print("코 → 하강뉴런:", " → ".join(이름표(x) for x in 경로1), f"| 시냅스 {len(경로1)-1}단계")
print("코 → 운동뉴런:", " → ".join(이름표(x) for x in 경로2), f"| 시냅스 {len(경로2)-1}단계")

**냄새에서 다리 근육까지 시냅스 세 번.** 후각 뉴런 → 촉각엽 투사뉴런 → 하강뉴런 → 척수 운동뉴런(`Tr extensor MN`, 다리 넙다리마디를 펴는 근육). 이 경로가 실제로 있다는 걸 표에서 직접 확인한 겁니다. 2024년 지도에는 척수가 없어서 여기까지 못 갔습니다. **이번 지도라서 되는 것**입니다.

물론 이건 "가장 짧은 길"이지 "실제로 쓰이는 길"이라는 뜻은 아닙니다. 그걸 알려면 4단계처럼 배선을 돌려 봐야 합니다.

## 4단계. 배선을 물통 모델로 돌려 보기
뉴런 하나를 물통이라고 생각합니다.
- 연결된 뉴런이 발화하면 물이 들어옵니다. 시냅스가 많을수록 많이. 억제성 뉴런(GABA)에서 오면 오히려 빠집니다.
- 물이 1을 넘으면 그 뉴런이 발화하고, 통을 비우고, 3틱 쉽니다.
- 감각 뉴런(코)에만 바깥에서 물을 부어 줍니다.

이게 2024년 Shiu 팀이 쓴 모델의 뼈대이고, 우리 실험실(초파리 뇌 오리)이 쓰는 것과 같습니다. **학습은 없습니다.** 배선은 고정입니다.

먼저 신경전달물질 표로 억제성 뉴런에 마이너스 부호를 붙이고, 뉴런마다 들어오는 가중치 합을 3으로 맞춥니다(폭주 방지).

In [ ]:
import scipy.sparse as sp
신경전달물질 = pd.read_feather("신경전달물질표.feather", columns=["body", "consensus_nt"]).drop_duplicates("body").set_index("body")["consensus_nt"]
부호 = np.where(신경전달물질.reindex(뉴런번호들).fillna("unclear").to_numpy() == "gaba", -1.0, 1.0).astype(np.float32)
print("억제성(GABA) 뉴런:", int((부호 < 0).sum()), "개")

N = len(뉴런번호들)
번호에서자리 = {int(b): i for i, b in enumerate(뉴런번호들)}
i = np.array([번호에서자리[int(a)] for a in 보내는쪽]); j = np.array([번호에서자리[int(b)] for b in 받는쪽])
w = 시냅스수.astype(np.float32) * 부호[i]
들어오는합 = np.zeros(N, np.float32); np.add.at(들어오는합, j, np.abs(w))
w = w * (3.0 / np.maximum(1.0, 들어오는합[j]))
배선 = sp.csr_matrix((w, (j, i)), shape=(N, N))   # 행 = 받는 뉴런, 열 = 보내는 뉴런
print("배선 행렬:", 배선.shape, "연결", f"{배선.nnz:,}")

채널을 정합니다. 냄새를 넣을 곳(왼쪽·오른쪽 후각 뉴런), 읽을 곳(촉각엽 투사뉴런 좌우, 하강뉴런, 운동뉴런).

In [ ]:
def 자리들(조건):
    return np.array([번호에서자리[int(b)] for b in 뉴런만[조건]["bodyId"] if int(b) in 번호에서자리])
좌우 = 뉴런만["rootSide"].fillna(뉴런만["somaSide"])
채널 = {
    "후각_왼쪽": 자리들((뉴런만["class"] == "olfactory") & (좌우 == "L")),
    "후각_오른쪽": 자리들((뉴런만["class"] == "olfactory") & (좌우 == "R")),
    "촉각엽_왼쪽": 자리들((뉴런만["class"] == "ALPN") & (좌우 == "L")),
    "촉각엽_오른쪽": 자리들((뉴런만["class"] == "ALPN") & (좌우 == "R")),
    "하강뉴런": 자리들(뉴런만["superclass"] == "descending_neuron"),
    "운동뉴런": 자리들(뉴런만["superclass"].isin(["vnc_motor", "cb_motor"])),
}
{k: len(v) for k, v in 채널.items()}

이제 물통 모델 본체입니다. 20줄입니다. `자극`은 채널 이름과 세기(0~1)입니다. 매 틱마다 (1) 물이 조금 새고(×0.94) (2) 직전 틱에 발화한 뉴런에서 물이 오고 (3) 감각 뉴런에 바깥 물을 붓고 (4) 1을 넘으면 발화합니다.

In [ ]:
def 물통모델(자극, 틱수=100, 이득=3.0, 씨앗=23):
    rng = np.random.default_rng(씨앗)
    물 = np.zeros(N, np.float32); 휴식 = np.zeros(N, np.int8); 발화 = np.zeros(N, bool)
    기록 = {k: [] for k in ["촉각엽_왼쪽", "촉각엽_오른쪽", "하강뉴런", "운동뉴런"]}; 전체 = []
    for t in range(틱수):
        물 *= 0.94; 물[휴식 > 0] = 0; 휴식[휴식 > 0] -= 1
        물 += (이득 / 3.0) * (배선 @ 발화.astype(np.float32))       # (2) 연결된 뉴런에서 물이 옴 (이득 3이 기본)
        for 이름, 세기 in 자극.items():                               # (3) 감각 뉴런에 바깥 물
            자리 = 채널[이름]; 맞음 = rng.random(len(자리)) < 0.6 * 세기
            물[자리[맞음]] += 1.1
        발화 = (물 >= 1) & (휴식 == 0)                                # (4) 넘치면 발화
        물[발화] = 0; 휴식[발화] = 3; 물 = np.maximum(물, -3)
        for k in 기록: 기록[k].append(발화[채널[k]].mean())
        전체.append(int(발화.sum()))
    return {k: np.array(v) for k, v in 기록.items()}, np.array(전체)

기록, 전체 = 물통모델({"후각_왼쪽": 0.8})
print("틱당 전체 발화(마지막 10틱 평균):", int(전체[-10:].mean()))
for k, v in 기록.items():
    print(f"{k:<10} 발화율(마지막 30틱 평균): {100*v[-30:].mean():.3f}%")

**왼쪽에 냄새를 넣었으니 왼쪽 촉각엽이 더 세게 발화해야** 합니다. 그리고 하강뉴런과 운동뉴런까지 신호가 내려갔는지 보세요. 이제 좌우를 바꿔 넣고 비교합니다. 실험실의 "더듬이 바꿔 끼우기"가 이겁니다.

In [ ]:
기록_오른쪽, _ = 물통모델({"후각_오른쪽": 0.8})
그림, 축 = plt.subplots(1, 2, figsize=(12, 4))
for 축하나, 기록물, 제목 in [(축[0], 기록, "왼쪽에 냄새"), (축[1], 기록_오른쪽, "오른쪽에 냄새")]:
    축하나.plot(100 * 기록물["촉각엽_왼쪽"], label="촉각엽 왼쪽"); 축하나.plot(100 * 기록물["촉각엽_오른쪽"], label="촉각엽 오른쪽")
    축하나.plot(100 * 기록물["하강뉴런"], label="하강뉴런"); 축하나.plot(100 * 기록물["운동뉴런"], label="운동뉴런(척수)")
    축하나.set_title(제목); 축하나.set_xlabel("틱"); 축하나.set_ylabel("발화율 (%)"); 축하나.legend()
plt.tight_layout(); plt.show()

왼쪽에 냄새를 넣으면 왼쪽 촉각엽이, 오른쪽에 넣으면 오른쪽이 높습니다. 배선이 좌우를 지킵니다. 우리 실험실은 이 좌우 차이를 "회전", 하강뉴런 발화율을 "전진"으로 번역해 오리 다리에 넘깁니다. 번역 상수는 사람이 정한 것이고, 뇌는 배운 게 아니라 복사한 것입니다.

그런데 위 숫자를 보면 **운동뉴런은 거의 0%**입니다. 냄새가 척수까지 잘 안 내려갑니다. 왜일까요. 시냅스 10개 이상 연결만 남겨 척수로 가는 길이 가늘어졐고, 이득 3은 뇌 전체가 폭주하지 않게 낮게 잡은 값이기 때문입니다.

**마지막 실험 — 이득을 올리면?** 시냅스 세기(이득)를 3, 3.5, 4, 5로 바꿔 봅니다. 두 가지를 동시에 보세요. 척수 운동뉴런까지 신호가 가는지, 그리고 촉각엽의 **좌우 차이가 유지되는지**.

In [ ]:
print(f"{'이득':>4} | {'전체 발화/틱':>10} | {'촉각엽 L':>8} {'촉각엽 R':>8} | {'하강뉴런':>8} | {'운동뉴런':>8}")
for 이득 in (3.0, 3.5, 4.0, 5.0):
    기록_, 전체_ = 물통모델({"후각_왼쪽": 0.8}, 이득=이득)
    평균 = lambda k: 100 * 기록_[k][-30:].mean()
    print(f"{이득:>4} | {int(전체_[-10:].mean()):>10,} | {평균('촉각엽_왼쪽'):>7.2f}% {평균('촉각엽_오른쪽'):>7.2f}% | {평균('하강뉴런'):>7.3f}% | {평균('운동뉴런'):>7.3f}%")

결과를 읽어 보세요. 이득을 올리면 척수 운동뉴런까지 신호가 내려갑니다(이득 4에서 약 2%). 그런데 같은 순간 **촉각엽의 좌우 차이가 사라집니다.** 이득 3에서는 왼쪽 10% 대 오른쪽 8%로 차이가 났는데, 이득 4에서는 15% 대 15%로 같아집니다. 뇌 전체가 배경 발화로 포화돼 "어느 쪽에서 냄새가 나는지"를 잃는 겁니다.

이것이 실험실 원작자가 이득을 4에서 3으로 낮춘 이유이고, 신경과학에서 "억제와 흥분의 균형"이 왜 중요한지의 축소판입니다. 신호가 세면 방향을 잃고, 약하면 몸까지 못 갑니다. 그 사이의 좁은 구간에서 뇌가 일합니다.

한 가지 더. 아래에서 `억제끄기 = True`로 바꾸면 억제성 뉴런의 마이너스 부호를 없앱니다. 브레이크 없는 뇌가 어떻게 되는지 보세요.

In [ ]:
억제끄기 = False
if 억제끄기:
    배선.data = np.abs(배선.data)
기록_, 전체_ = 물통모델({"후각_왼쪽": 0.8})
print("틱당 전체 발화(마지막 10틱 평균):", int(전체_[-10:].mean()), "| 하강뉴런 발화율:", f"{100*기록_['하강뉴런'][-30:].mean():.3f}%")

## 정리
- 지도는 **표**입니다. 뉴런 표, 연결 표, 신경전달물질 표. 여러분이 직접 열었습니다.
- 냄새에서 다리 근육까지 시냅스 세 번. 척수가 있는 이번 지도라서 찾을 수 있었습니다.
- 물통 모델 20줄로 배선이 좌우를 지키는 것을 봤습니다. 학습은 없습니다. **복사한 뇌**입니다.
- 다음 편에서는 이 배선 위에 **보상(도파민 뉴런 340개)**을 얹어 배우게 하는 법, 그리고 그 보상을 설계하는 일이 왜 다음 시대의 일자리인지 다룹니다.

이 뇌가 2D 초파리를 걷게 하는 심플 판(설치 없음, 13MB): https://wonseokjayjung-fly-walk.static.hf.space/
로봇 오리 마이크로덕을 조종하는 3D 판: https://wonseokjayjung-fly-duck.static.hf.space/ (수컷 뇌 선택)

— AI 멘토 제이 · CONNECT AI LAB · AI가 만드는 기회를 소수가 독점하지 않도록.